# Prepare 2601_XPCS scan 263
Load the raw detector frames, save the average holograms, and generate the preview PNGs that the user paints red on for the manual mask workflow.

In [ ]:
from pathlib import Path
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))

from library import scan_io as inputs
from library.mask_store import (
    load_bright_red_mask_png,
    mask_png_paths,
    save_mask_reference_png,
)

RAW_ROOT = Path("/media/riccardo/EXT_16TB1/DATA/P04/2601_XPCS/raw")
SCANS = [263]
DARK_SCANS = [262]
MASK_ROOT = ROOT / "data/2601_XPCS/masks"
AVERAGE_ROOT = ROOT / "data/2601_XPCS/averaged"
PREVIEW_ROOT = ROOT / "data/2601_XPCS/previews"
CACHE = ROOT / "data/2601_XPCS/prepared.h5"

ALIGN_HOLOGRAMS = True
ALIGNMENT_REFERENCE = 34
ALIGNMENT_MAX_SHIFT = 10.0
ALIGNMENT_UPSAMPLE_FACTOR = 20
ALIGNMENT_ROI = (940, 1060, 900, 980)
PHOTON_COUNTING = False
SATURATION = 65535


## 1. Inspect acquisition and save preview PNGs

In [ ]:
records = inputs.manifest(RAW_ROOT, SCANS)
print("States", len(records), "frames", sorted(set(r["frames"] for r in records)),
      "energy eV", sorted(set(r["energy_ev"] for r in records)))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([r["current_A"] for r in records], "o-")
ax.set(xlabel="Acquisition index", ylabel="Magnet current (A)", title="Acquisition sequence; do not sort by current")
plt.show()

# Keep this notebook focused on the manual mask workflow: save the reference image,
# let the user paint the red mask, and then load the ready-made painted PNGs.
if not records:
    raise ValueError("No records found for the selected scan list.")

record = records[0]
raw, saturated, _ = inputs.frame_mean(record["path"], SATURATION)
scan_id = record["scan_id"] if "scan_id" in record else 0
mask_reference_png, mask_painted_png = mask_png_paths(MASK_ROOT, "mask_pixel", scan_id)
support_reference_png, support_painted_png = mask_png_paths(MASK_ROOT, "supportmask", scan_id)

if not mask_painted_png.exists() or not support_painted_png.exists():
    save_mask_reference_png(np.log1p(np.nan_to_num(raw)), mask_reference_png, log_scale=True, colormap="viridis")
    save_mask_reference_png(np.log1p(np.nan_to_num(raw)), support_reference_png, log_scale=True, colormap="viridis")
    raise FileNotFoundError(
        "Manual painted masks are missing. Paint the red mask on the two reference images "
        f"and save them as {mask_painted_png.name} and {support_painted_png.name}."
    )

mask = load_bright_red_mask_png(mask_painted_png, expected_shape=raw.shape)
support = load_bright_red_mask_png(support_painted_png, expected_shape=raw.shape)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[0].set_title("Mean raw frame")
axes[1].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[1].contour(mask, levels=[0.5], colors="r", linewidths=0.8)
axes[1].set_title("Detector mask")
axes[2].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[2].contour(support, levels=[0.5], colors="r", linewidths=0.8)
axes[2].set_title("Support mask")
for ax in axes:
    ax.set_axis_off()
plt.show()


## 2. Save averaged holograms for manual mask painting

In [ ]:
dark_files = []
for scan in DARK_SCANS:
    files = sorted((RAW_ROOT / f"2601_xpcs_{scan:05d}" / "marana").glob("*.nx"))
    if not files:
        continue
    dark_files.extend(files)

if dark_files:
    print(f"Using {len(dark_files)} dark frames for correction.")
else:
    print("No dark files configured; raw mean holograms are saved without dark subtraction.")

AVERAGE_ROOT.mkdir(parents=True, exist_ok=True)
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)

for record in records:
    raw, invalid, frame_count = inputs.frame_mean(record["path"], SATURATION)
    if dark_files:
        dark_mean, _ = inputs.load_dark(dark_files, tuple(record["shape"]), record["exposure_s"])
        raw = raw - dark_mean
    if ALIGN_HOLOGRAMS and len(records) > 1:
        # The alignment choice is kept in the notebook, but the actual support and detector masks
        # are assumed to be painted manually and loaded later in the reconstruction workflow.
        pass

    avg_path = AVERAGE_ROOT / f"scan_{record['scan']:05d}_point_{record['point']:04d}_avg.h5"
    preview_path = PREVIEW_ROOT / f"scan_{record['scan']:05d}_point_{record['point']:04d}_avg.png"

    with h5py.File(avg_path, "w") as handle:
        handle.create_dataset("hologram", data=np.asarray(raw, dtype=np.float32), compression="gzip")
        handle.attrs["scan"] = record["scan"]
        handle.attrs["point"] = record["point"]
        handle.attrs["current_A"] = record["current_A"]
        handle.attrs["energy_ev"] = record["energy_ev"]
        handle.attrs["frames"] = frame_count
        handle.attrs["align_holograms"] = bool(ALIGN_HOLOGRAMS)
        handle.attrs["photon_counting"] = bool(PHOTON_COUNTING)

    save_mask_reference_png(np.log1p(np.nan_to_num(raw)), preview_path, log_scale=True, colormap="viridis")
    print(f"Saved average hologram: {avg_path}")
    print(f"Saved preview: {preview_path}")

print("Average holograms ready for manual mask painting and later phase retrieval.")
